In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP & LIBRARY IMPORTS
# ============================================================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip uninstall -y prophet cmdstanpy -qq
!pip install -q prophet xgboost joblib

# Core imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import os
import json
import joblib
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import TimeSeriesSplit

# Forecasting
from prophet import Prophet
from xgboost import XGBRegressor

# Configuration
OUTPUT_DIR = '/content/drive/MyDrive/DATA_SCIENCE_PROJECT/'
MODELS_DIR = f'{OUTPUT_DIR}models/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('='*60)
print('✅ ENVIRONMENT CONFIGURED')
print('='*60)
print(f'📁 Output: {OUTPUT_DIR}')
print(f'🔧 Models: {MODELS_DIR}')
print(f'🎲 Random Seed: {RANDOM_SEED}')

In [ ]:
# ============================================================
# CELL 2: DATA LOADING & CLEANING PIPELINE
# ============================================================

print('='*60)
print('🧹 DATA LOADING & CLEANING')
print('='*60)

# Load data
file_path = '/content/drive/MyDrive/DATA_SCIENCE_PROJECT/Online-Retail.csv'
df_raw = pd.read_csv(file_path, encoding='latin-1', parse_dates=['InvoiceDate'])
initial_rows = len(df_raw)

print(f'\n📊 Raw data loaded: {initial_rows:,} rows')

# Cleaning pipeline
df = df_raw.copy()

# Step 1: Remove duplicates
df = df.drop_duplicates()

# Step 2: Drop nulls in critical columns
df = df.dropna(subset=['CustomerID', 'Description'])

# Step 3: Remove invalid stock codes
invalid_codes = ['POST', 'BANK CHARGES', 'C2', 'M', 'AMAZONFEE', 'TEST', 'MANUAL', 'D']
df = df[~df['StockCode'].str.upper().isin(invalid_codes)]
df = df[~df['StockCode'].str.match(r'^[A-Za-z]+$', na=False)]

# Step 4: Keep positive transactions only
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Step 5: Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Step 6: Winsorize outliers (IQR method with 3x multiplier)
for col in ['Quantity', 'UnitPrice', 'TotalPrice']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(lower=Q1 - 3*IQR, upper=Q3 + 3*IQR)

# Step 7: Convert CustomerID to int
df['CustomerID'] = df['CustomerID'].astype(int)

# Summary
print(f'\n✅ CLEANING COMPLETE')
print(f'   Original: {initial_rows:,} rows')
print(f'   Cleaned: {len(df):,} rows ({len(df)/initial_rows*100:.1f}% retained)')
print(f'   Date range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')
print(f'   Customers: {df["CustomerID"].nunique():,}')
print(f'   Products: {df["StockCode"].nunique():,}')
print(f'   Revenue: £{df["TotalPrice"].sum():,.2f}')

# Save cleaned data
df.to_csv(f'{OUTPUT_DIR}cleaned_data.csv', index=False)
print(f'\n💾 Saved: {OUTPUT_DIR}cleaned_data.csv')

In [ ]:
# ============================================================
# CELL 3: RFM FEATURE ENGINEERING
# ============================================================

print('='*60)
print('🔬 RFM FEATURE ENGINEERING')
print('='*60)

# Reference date
ref_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f'\n📅 Reference date: {ref_date}')

# Core RFM features
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (ref_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                               # Frequency
    'TotalPrice': 'sum'                                   # Monetary
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalPrice': 'Monetary'
})

# Additional features
rfm['AvgOrderValue'] = rfm['Monetary'] / rfm['Frequency']
rfm['ProductDiversity'] = df.groupby('CustomerID')['StockCode'].nunique()
rfm['AvgBasketSize'] = df.groupby('CustomerID')['Quantity'].mean()

# CLV Proxy
rfm['RecencyMonths'] = (rfm['Recency'] / 30.44).clip(lower=0.1)
rfm['CLV_Proxy'] = rfm['Frequency'] * rfm['AvgOrderValue'] / rfm['RecencyMonths']

# Remove extreme outliers using Z-score
from scipy import stats
z_scores = np.abs(stats.zscore(rfm.select_dtypes(include=[np.number])))
rfm = rfm[(z_scores < 4).all(axis=1)]

print(f'\n✅ Features created: {len(rfm.columns)}')
print(f'   Customers: {len(rfm):,}')
print(f'\n📋 Feature Summary:')
display(rfm.describe().round(2))

# Save features
rfm.to_csv(f'{OUTPUT_DIR}customer_features.csv')
print(f'\n💾 Saved: {OUTPUT_DIR}customer_features.csv')

In [ ]:
# ============================================================
# CELL 4: CLUSTERING - K-MEANS WITH OPTIMAL K SELECTION
# ============================================================

print('='*60)
print('🎯 CUSTOMER SEGMENTATION: K-MEANS CLUSTERING')
print('='*60)

# Features for clustering
cluster_features = ['Recency', 'Frequency', 'Monetary']
X_cluster = rfm[cluster_features].values

# CRITICAL: Fit and SAVE the scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Save scaler for production
joblib.dump(scaler, f'{MODELS_DIR}rfm_scaler.joblib')
print(f'✅ Scaler saved: {MODELS_DIR}rfm_scaler.joblib')

# Find optimal K
print(f'\n🔍 Finding optimal K...')
k_range = range(2, 11)
metrics = {'k': [], 'SSE': [], 'Silhouette': [], 'DBI': [], 'CHI': []}

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_SEED)
    labels = kmeans.fit_predict(X_scaled)
    
    metrics['k'].append(k)
    metrics['SSE'].append(kmeans.inertia_)
    metrics['Silhouette'].append(silhouette_score(X_scaled, labels))
    metrics['DBI'].append(davies_bouldin_score(X_scaled, labels))
    metrics['CHI'].append(calinski_harabasz_score(X_scaled, labels))

metrics_df = pd.DataFrame(metrics)

# Select optimal K (highest silhouette)
optimal_k = metrics_df.loc[metrics_df['Silhouette'].idxmax(), 'k']
print(f'\n🎯 Optimal K = {optimal_k} (Silhouette: {metrics_df["Silhouette"].max():.4f})')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(metrics['k'], metrics['SSE'], 'bo-', linewidth=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('SSE')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].axvline(x=optimal_k, color='r', linestyle='--', alpha=0.7)

axes[1].plot(metrics['k'], metrics['Silhouette'], 'go-', linewidth=2)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].axvline(x=optimal_k, color='r', linestyle='--', alpha=0.7)

axes[2].plot(metrics['k'], metrics['DBI'], 'ro-', linewidth=2)
axes[2].set_xlabel('K'); axes[2].set_ylabel('Davies-Bouldin Index')
axes[2].set_title('Davies-Bouldin Index (lower=better)', fontweight='bold')
axes[2].axvline(x=optimal_k, color='r', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}clustering_optimization.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\n💾 Saved: {OUTPUT_DIR}clustering_optimization.png')

In [ ]:
# ============================================================
# CELL 5: TRAIN FINAL CLUSTERING MODEL & SAVE
# ============================================================

print('='*60)
print('🏋️ TRAINING FINAL K-MEANS MODEL')
print('='*60)

# Train final model with more initializations for stability
kmeans_final = KMeans(
    n_clusters=int(optimal_k),
    init='k-means++',
    n_init=30,  # More inits for production
    max_iter=500,
    random_state=RANDOM_SEED
)
rfm['Cluster'] = kmeans_final.fit_predict(X_scaled)

# CRITICAL: Save the trained model
joblib.dump(kmeans_final, f'{MODELS_DIR}kmeans_model_customer_categorization.joblib')
print(f'✅ K-Means model saved: {MODELS_DIR}kmeans_model_customer_categorization.joblib')

# Cluster summary
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'CLV_Proxy': 'mean'
}).round(2)
cluster_summary['Size'] = rfm.groupby('Cluster').size()
cluster_summary['Percentage'] = (cluster_summary['Size'] / len(rfm) * 100).round(1)

# Assign business labels based on RFM
def assign_label(row):
    r, f, m = row['Recency'], row['Frequency'], row['Monetary']
    r_pct = 100 - (r / rfm['Recency'].max() * 100)
    f_pct = (f / rfm['Frequency'].max() * 100)
    m_pct = (m / rfm['Monetary'].max() * 100)
    
    if r_pct > 60 and f_pct > 50 and m_pct > 50:
        return 'Champions'
    elif f_pct > 40 and m_pct > 40:
        return 'Loyal'
    elif r_pct < 30:
        return 'Lost'
    elif r_pct < 50:
        return 'At Risk'
    else:
        return 'Potential'

cluster_summary['Label'] = cluster_summary.apply(assign_label, axis=1)

print(f'\n📊 CLUSTER SUMMARY:')
display(cluster_summary)

# Final metrics
final_silhouette = silhouette_score(X_scaled, rfm['Cluster'])
final_dbi = davies_bouldin_score(X_scaled, rfm['Cluster'])

print(f'\n📈 FINAL MODEL METRICS:')
print(f'   Silhouette Score: {final_silhouette:.4f}')
print(f'   Davies-Bouldin Index: {final_dbi:.4f}')
print(f'   Number of Clusters: {int(optimal_k)}')

# Save results
rfm.to_csv(f'{OUTPUT_DIR}customer_segments.csv')
cluster_summary.to_csv(f'{OUTPUT_DIR}cluster_summary.csv')

print(f'\n💾 Saved:')
print(f'   - {OUTPUT_DIR}customer_segments.csv')
print(f'   - {OUTPUT_DIR}cluster_summary.csv')

In [ ]:
# ============================================================
# CELL 6: CLUSTER VISUALIZATION
# ============================================================

print('='*60)
print('📊 CLUSTER VISUALIZATION')
print('='*60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Cluster Distribution
colors = plt.cm.tab10(np.linspace(0, 1, int(optimal_k)))
bars = axes[0].bar(cluster_summary.index, cluster_summary['Size'], color=colors)
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Customer Distribution by Cluster', fontweight='bold')
for bar, label, pct in zip(bars, cluster_summary['Label'], cluster_summary['Percentage']):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{label}\n{pct:.1f}%', ha='center', va='bottom', fontsize=9)

# 2. RFM Heatmap
rfm_normalized = cluster_summary[['Recency', 'Frequency', 'Monetary']].copy()
rfm_normalized['Recency'] = 1 / (rfm_normalized['Recency'] + 1)  # Invert for visualization
sns.heatmap(rfm_normalized.T, annot=True, fmt='.0f', cmap='RdYlGn', ax=axes[1])
axes[1].set_title('RFM Profile by Cluster', fontweight='bold')

# 3. 2D Scatter (Recency vs Monetary)
scatter = axes[2].scatter(rfm['Recency'], rfm['Monetary'],
                          c=rfm['Cluster'], cmap='tab10', alpha=0.6, s=20)
axes[2].set_xlabel('Recency (days)')
axes[2].set_ylabel('Monetary (£)')
axes[2].set_title('Customer Segments: Recency vs Monetary', fontweight='bold')
plt.colorbar(scatter, ax=axes[2], label='Cluster')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}customer_segments_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\n💾 Saved: {OUTPUT_DIR}customer_segments_visualization.png')

In [ ]:
# ============================================================
# CELL 7: PREPARE TIME-SERIES DATA FOR FORECASTING
# ============================================================

print('='*60)
print('📈 PREPARING TIME-SERIES DATA')
print('='*60)

# Aggregate daily sales
daily_sales = df.groupby(df['InvoiceDate'].dt.date)['TotalPrice'].sum().reset_index()
daily_sales.columns = ['Date', 'Sales']
daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])

# Fill missing dates
date_range = pd.date_range(start=daily_sales['Date'].min(), end=daily_sales['Date'].max(), freq='D')
daily_sales = pd.DataFrame({'Date': date_range}).merge(daily_sales, on='Date', how='left')
daily_sales['Sales'] = daily_sales['Sales'].fillna(0)

# Create features for XGBoost
daily_sales['DayOfWeek'] = daily_sales['Date'].dt.dayofweek
daily_sales['Month'] = daily_sales['Date'].dt.month
daily_sales['DayOfMonth'] = daily_sales['Date'].dt.day
daily_sales['WeekOfYear'] = daily_sales['Date'].dt.isocalendar().week.astype(int)

# Lag features
for lag in [1, 7, 14, 28]:
    daily_sales[f'Lag_{lag}'] = daily_sales['Sales'].shift(lag)

# Rolling features
for window in [7, 14, 30]:
    daily_sales[f'Rolling_Mean_{window}'] = daily_sales['Sales'].rolling(window).mean()
    daily_sales[f'Rolling_Std_{window}'] = daily_sales['Sales'].rolling(window).std()

# Drop NaN rows
daily_sales = daily_sales.dropna().reset_index(drop=True)

print(f'\n✅ Time-series prepared')
print(f'   Total days: {len(daily_sales)}')
print(f'   Date range: {daily_sales["Date"].min()} to {daily_sales["Date"].max()}')
print(f'   Features: {len([c for c in daily_sales.columns if c not in ["Date", "Sales"]])}')
print(f'   Avg daily sales: £{daily_sales["Sales"].mean():,.2f}')

# Train/Test split (last 30 days for testing)
TEST_SIZE = 30
train_data = daily_sales.iloc[:-TEST_SIZE].copy()
test_data = daily_sales.iloc[-TEST_SIZE:].copy()

print(f'\n📊 Train/Test Split:')
print(f'   Train: {len(train_data)} days')
print(f'   Test: {len(test_data)} days')

# Save for later use
daily_sales.to_csv(f'{OUTPUT_DIR}daily_sales_features.csv', index=False)
print(f'\n💾 Saved: {OUTPUT_DIR}daily_sales_features.csv')

In [ ]:
# ============================================================
# CELL 8: TRAIN PROPHET MODEL
# ============================================================

print('='*60)
print('🤖 MODEL 1: FACEBOOK PROPHET')
print('='*60)

# Prepare Prophet data
prophet_train = train_data[['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'})
prophet_test = test_data[['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'})

# Cross-validation
print(f'\n🔬 Cross-validation (4 folds)...')
tscv = TimeSeriesSplit(n_splits=4)
prophet_cv_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(train_data), 1):
    cv_train = prophet_train.iloc[train_idx]
    cv_val = prophet_train.iloc[val_idx]
    
    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode='multiplicative'
    )
    model.add_country_holidays(country_name='UK')
    model.fit(cv_train)
    
    forecast = model.predict(cv_val[['ds']])
    mape = np.mean(np.abs((cv_val['y'].values - forecast['yhat'].values) / cv_val['y'].values)) * 100
    prophet_cv_scores.append(mape)
    print(f'   Fold {fold}: MAPE = {mape:.2f}%')

prophet_cv_mape = np.mean(prophet_cv_scores)
print(f'\n   CV Average MAPE: {prophet_cv_mape:.2f}%')

# Train final model
print(f'\n🏋️ Training final Prophet model...')
prophet_model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
    seasonality_mode='multiplicative',
    interval_width=0.95
)
prophet_model.add_country_holidays(country_name='UK')
prophet_model.fit(prophet_train)

# Test predictions
prophet_forecast = prophet_model.predict(prophet_test[['ds']])
prophet_pred = prophet_forecast['yhat'].values

# Test metrics
prophet_test_mape = np.mean(np.abs((prophet_test['y'].values - prophet_pred) / prophet_test['y'].values)) * 100
prophet_test_rmse = np.sqrt(mean_squared_error(prophet_test['y'].values, prophet_pred))
prophet_test_r2 = r2_score(prophet_test['y'].values, prophet_pred)

print(f'\n📈 Test Performance:')
print(f'   MAPE: {prophet_test_mape:.2f}%')
print(f'   RMSE: £{prophet_test_rmse:,.2f}')
print(f'   R²: {prophet_test_r2:.4f}')

# Save model
joblib.dump(prophet_model, f'{MODELS_DIR}prophet_model_sales_forecast.joblib')
print(f'\n✅ Prophet model saved: {MODELS_DIR}prophet_model_sales_forecast.joblib')

In [ ]:
# ============================================================
# CELL 9: TRAIN XGBOOST MODEL
# ============================================================

print('='*60)
print('🚀 MODEL 2: XGBOOST REGRESSOR')
print('='*60)

# Prepare features
feature_cols = [c for c in train_data.columns if c not in ['Date', 'Sales']]
X_train = train_data[feature_cols]
y_train = train_data['Sales']
X_test = test_data[feature_cols]
y_test = test_data['Sales']

print(f'\n📊 Features: {len(feature_cols)}')

# Cross-validation
print(f'\n🔬 Cross-validation (4 folds)...')
xgb_cv_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(train_data), 1):
    cv_X_train = X_train.iloc[train_idx]
    cv_y_train = y_train.iloc[train_idx]
    cv_X_val = X_train.iloc[val_idx]
    cv_y_val = y_train.iloc[val_idx]
    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        early_stopping_rounds=20,
        n_jobs=-1
    )
    model.fit(cv_X_train, cv_y_train, eval_set=[(cv_X_val, cv_y_val)], verbose=False)
    
    pred = model.predict(cv_X_val)
    mape = np.mean(np.abs((cv_y_val.values - pred) / cv_y_val.values)) * 100
    xgb_cv_scores.append(mape)
    print(f'   Fold {fold}: MAPE = {mape:.2f}%')

xgb_cv_mape = np.mean(xgb_cv_scores)
print(f'\n   CV Average MAPE: {xgb_cv_mape:.2f}%')

# Train final model
print(f'\n🏋️ Training final XGBoost model...')
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train, verbose=False)

# Test predictions
xgb_pred = xgb_model.predict(X_test)

# Test metrics
xgb_test_mape = np.mean(np.abs((y_test.values - xgb_pred) / y_test.values)) * 100
xgb_test_rmse = np.sqrt(mean_squared_error(y_test.values, xgb_pred))
xgb_test_r2 = r2_score(y_test.values, xgb_pred)

print(f'\n📈 Test Performance:')
print(f'   MAPE: {xgb_test_mape:.2f}%')
print(f'   RMSE: £{xgb_test_rmse:,.2f}')
print(f'   R²: {xgb_test_r2:.4f}')

# Feature importance
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f'\n🔍 Top 5 Important Features:')
display(importance.head())

# Save model
joblib.dump(xgb_model, f'{MODELS_DIR}xgboost_model_sales_forecast.joblib')
print(f'\n✅ XGBoost model saved: {MODELS_DIR}xgboost_model_sales_forecast.joblib')

In [ ]:
# ============================================================
# CELL 10: MODEL COMPARISON & ENSEMBLE
# ============================================================

print('='*60)
print('🏆 MODEL COMPARISON & ENSEMBLE')
print('='*60)

# Comparison table
comparison = pd.DataFrame({
    'Model': ['Prophet', 'XGBoost'],
    'CV_MAPE': [prophet_cv_mape, xgb_cv_mape],
    'Test_MAPE': [prophet_test_mape, xgb_test_mape],
    'Test_RMSE': [prophet_test_rmse, xgb_test_rmse],
    'Test_R2': [prophet_test_r2, xgb_test_r2]
})

print(f'\n📊 MODEL PERFORMANCE:')
display(comparison.round(4))

# Create ensemble (weighted by inverse MAPE)
w_prophet = 1 / prophet_test_mape
w_xgb = 1 / xgb_test_mape
total_weight = w_prophet + w_xgb

ensemble_pred = (w_prophet/total_weight * prophet_pred + w_xgb/total_weight * xgb_pred)

# Ensemble metrics
ensemble_mape = np.mean(np.abs((y_test.values - ensemble_pred) / y_test.values)) * 100
ensemble_rmse = np.sqrt(mean_squared_error(y_test.values, ensemble_pred))
ensemble_r2 = r2_score(y_test.values, ensemble_pred)

print(f'\n🎯 ENSEMBLE PERFORMANCE:')
print(f'   Weights: Prophet={w_prophet/total_weight:.2%}, XGBoost={w_xgb/total_weight:.2%}')
print(f'   MAPE: {ensemble_mape:.2f}%')
print(f'   RMSE: £{ensemble_rmse:,.2f}')
print(f'   R²: {ensemble_r2:.4f}')

# Determine best approach
all_mapes = [prophet_test_mape, xgb_test_mape, ensemble_mape]
best_idx = np.argmin(all_mapes)
best_names = ['Prophet', 'XGBoost', 'Ensemble']
best_model_name = best_names[best_idx]

print(f'\n🏆 BEST MODEL: {best_model_name} (MAPE: {all_mapes[best_idx]:.2f}%)')

# Save comparison
comparison_full = comparison.copy()
comparison_full = pd.concat([comparison_full, pd.DataFrame({
    'Model': ['Ensemble'],
    'CV_MAPE': [np.nan],
    'Test_MAPE': [ensemble_mape],
    'Test_RMSE': [ensemble_rmse],
    'Test_R2': [ensemble_r2]
})], ignore_index=True)

comparison_full.to_csv(f'{OUTPUT_DIR}model_comparison.csv', index=False)
print(f'\n💾 Saved: {OUTPUT_DIR}model_comparison.csv')

In [ ]:
# ============================================================
# CELL 11: FORECAST VISUALIZATION
# ============================================================

print('='*60)
print('📊 FORECAST VISUALIZATION')
print('='*60)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Sales Forecasting: Model Comparison', fontsize=14, fontweight='bold')

# 1. Test Set Predictions
ax = axes[0, 0]
ax.plot(test_data['Date'], y_test.values, 'ko-', label='Actual', linewidth=2, markersize=4)
ax.plot(test_data['Date'], prophet_pred, 'b--', label='Prophet', linewidth=1.5, alpha=0.7)
ax.plot(test_data['Date'], xgb_pred, 'r--', label='XGBoost', linewidth=1.5, alpha=0.7)
ax.plot(test_data['Date'], ensemble_pred, 'g-', label='Ensemble', linewidth=2)
ax.set_xlabel('Date')
ax.set_ylabel('Sales (£)')
ax.set_title('Test Set: Model Predictions vs Actual', fontweight='bold')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# 2. Model Performance Comparison
ax = axes[0, 1]
models = ['Prophet', 'XGBoost', 'Ensemble']
mapes = [prophet_test_mape, xgb_test_mape, ensemble_mape]
colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = ax.bar(models, mapes, color=colors, alpha=0.7)
ax.set_ylabel('MAPE (%)')
ax.set_title('Model MAPE Comparison (lower is better)', fontweight='bold')
for bar, mape in zip(bars, mapes):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{mape:.2f}%',
            ha='center', va='bottom', fontweight='bold')

# 3. Residual Analysis
ax = axes[1, 0]
ax.scatter(y_test.values, ensemble_pred, alpha=0.6, c='green')
max_val = max(y_test.max(), ensemble_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')
ax.set_xlabel('Actual Sales (£)')
ax.set_ylabel('Predicted Sales (£)')
ax.set_title('Ensemble: Actual vs Predicted', fontweight='bold')
ax.legend()

# 4. Error Distribution
ax = axes[1, 1]
residuals = y_test.values - ensemble_pred
ax.hist(residuals, bins=15, color='#2ecc71', alpha=0.7, edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Prediction Error (£)')
ax.set_ylabel('Frequency')
ax.set_title('Ensemble: Error Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}forecast_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\n💾 Saved: {OUTPUT_DIR}forecast_comparison.png')

In [ ]:
# ============================================================
# CELL 12: GENERATE FUTURE FORECAST & DASHBOARD DATA
# ============================================================

print('='*60)
print('🔮 GENERATING FUTURE FORECAST & DASHBOARD DATA')
print('='*60)

# Generate 30-day future forecast using Prophet
future = prophet_model.make_future_dataframe(periods=30)
future_forecast = prophet_model.predict(future)
forecast_30days = future_forecast.tail(30)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()

print(f'\n📅 30-Day Forecast Generated')
print(f'   Start: {forecast_30days["ds"].min()}')
print(f'   End: {forecast_30days["ds"].max()}')
print(f'   Avg daily sales: £{forecast_30days["yhat"].mean():,.2f}')
print(f'   Total 30-day projection: £{forecast_30days["yhat"].sum():,.2f}')

# Create dashboard data
dashboard_data = {
    'project_info': {
        'title': 'E-commerce Customer Segmentation & Sales Forecasting',
        'authors': ['Sereno', 'Page', 'Laudato', 'Dulce'],
        'instructor': 'Sir Charlston Sean Gono',
        'date': datetime.now().strftime('%Y-%m-%d')
    },
    'summary': {
        'total_customers': int(len(rfm)),
        'total_revenue': float(df['TotalPrice'].sum()),
        'num_clusters': int(optimal_k),
        'avg_daily_sales': float(daily_sales['Sales'].mean()),
        'best_forecast_model': best_model_name,
        'best_mape': float(all_mapes[best_idx]),
        'silhouette_score': float(final_silhouette)
    },
    'cluster_summary': cluster_summary.to_dict(),
    'model_performance': {
        'prophet': {'cv_mape': prophet_cv_mape, 'test_mape': prophet_test_mape, 'r2': prophet_test_r2},
        'xgboost': {'cv_mape': xgb_cv_mape, 'test_mape': xgb_test_mape, 'r2': xgb_test_r2},
        'ensemble': {'test_mape': ensemble_mape, 'r2': ensemble_r2}
    },
    'forecast_30days': [
        {
            'ds': row['ds'].strftime('%Y-%m-%d'),
            'yhat': float(row['yhat']),
            'yhat_lower': float(row['yhat_lower']),
            'yhat_upper': float(row['yhat_upper'])
        }
        for _, row in forecast_30days.iterrows()
    ]
}

# Save dashboard data
with open(f'{OUTPUT_DIR}dashboard_data.json', 'w') as f:
    json.dump(dashboard_data, f, indent=2)

print(f'\n💾 Dashboard data saved: {OUTPUT_DIR}dashboard_data.json')

In [ ]:
# ============================================================
# CELL 13: FINAL SUMMARY & DEPLOYMENT ARTIFACTS
# ============================================================

print('='*70)
print('✅ TRAINING PIPELINE COMPLETE')
print('='*70)

print(f'\n📁 OUTPUT DIRECTORY: {OUTPUT_DIR}')
print(f'\n🔧 DEPLOYMENT ARTIFACTS (MODELS):')
print(f'   1. {MODELS_DIR}rfm_scaler.joblib')
print(f'   2. {MODELS_DIR}kmeans_model_customer_categorization.joblib')
print(f'   3. {MODELS_DIR}prophet_model_sales_forecast.joblib')
print(f'   4. {MODELS_DIR}xgboost_model_sales_forecast.joblib')

print(f'\n📊 DATA FILES:')
print(f'   1. {OUTPUT_DIR}cleaned_data.csv')
print(f'   2. {OUTPUT_DIR}customer_features.csv')
print(f'   3. {OUTPUT_DIR}customer_segments.csv')
print(f'   4. {OUTPUT_DIR}cluster_summary.csv')
print(f'   5. {OUTPUT_DIR}daily_sales_features.csv')
print(f'   6. {OUTPUT_DIR}model_comparison.csv')
print(f'   7. {OUTPUT_DIR}dashboard_data.json')

print(f'\n📈 VISUALIZATIONS:')
print(f'   1. {OUTPUT_DIR}clustering_optimization.png')
print(f'   2. {OUTPUT_DIR}customer_segments_visualization.png')
print(f'   3. {OUTPUT_DIR}forecast_comparison.png')

print(f'\n🎯 MODEL PERFORMANCE SUMMARY:')
print(f'\n   CUSTOMER SEGMENTATION (K-Means):')
print(f'   ├─ Clusters: {int(optimal_k)}')
print(f'   ├─ Silhouette Score: {final_silhouette:.4f}')
print(f'   └─ Davies-Bouldin Index: {final_dbi:.4f}')

print(f'\n   SALES FORECASTING:')
print(f'   ├─ Prophet: MAPE={prophet_test_mape:.2f}%, R²={prophet_test_r2:.4f}')
print(f'   ├─ XGBoost: MAPE={xgb_test_mape:.2f}%, R²={xgb_test_r2:.4f}')
print(f'   └─ Ensemble: MAPE={ensemble_mape:.2f}%, R²={ensemble_r2:.4f}')
print(f'   └─ Best: {best_model_name}')

print(f'\n🚀 READY FOR DEPLOYMENT!')
print(f'\n   To use models in production:')
print(f'   >>> import joblib')
print(f'   >>> scaler = joblib.load("{MODELS_DIR}rfm_scaler.joblib")')
print(f'   >>> kmeans = joblib.load("{MODELS_DIR}kmeans_model_customer_categorization.joblib")')
print(f'   >>> prophet = joblib.load("{MODELS_DIR}prophet_model_sales_forecast.joblib")')

print(f'\n' + '='*70)
print('🎓 PIPELINE EXECUTION COMPLETE')
print('='*70)